## Cleaning steps

1. Remove duplicates
2. Remove all row nulls
3. Fill the nulls/''/' '  
4. Trim spaces
5. Standardize case
6. Cast data types
7. Filter invalid rows
7. Rename columns
9. Add metadata columns if applicable
10. Validate schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

## Load the table

In [0]:
df = spark.table('bikes_catalog.bronze.bronze_cust_az12')
df.show()

## Cleaning

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.na.drop(how='all')

In [0]:
for col in df.columns:
    print(f'Nulls in {col}: {df.filter(F.col(col).isNull()).count()}')

In [0]:
df = df.na.fill({'GEN': 'Unknown'})

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

df.show()

In [0]:
df.select('GEN').distinct().show()

In [0]:
df.filter(df.GEN == '').show()

In [0]:
df = df.withColumn(
    'GEN',
    F.when(df.GEN == '', 'Unknown')
     .when(df.GEN  == 'Female', 'F')
     .when(df.GEN  == 'Male', 'M')
     .otherwise(df.GEN)
)
df.show()

In [0]:
df.select('GEN').distinct().show()

In [0]:
df = df.withColumn('CID', F.regexp_replace('CID', 'NAS', ''))
df.show()

In [0]:
df = df.withColumnRenamed('GEN','Gender')
df.show()

In [0]:
df = df.withColumn("BDATE", F.to_date("BDATE"))

In [0]:
df.printSchema()

In [0]:
df.write.mode('overwrite').saveAsTable('bikes_catalog.silver.silver_cust_az12')